In [ ]:
import os
import torch
import numpy as np 
from spikydiff.plotting import set_nice_params, get_next_panel_label


from local_memorization.scripts.cifar_color.loadplot_local_coverage import get_data, \
      plot_local_dominance, plot_memo_tau, \
      plot_data_sparsity, plot_data_sparsity_per_class, \
      plot_local_dominance_per_class, plot_memo_tau_per_class, \
      load_samples, plot_samples_with_closest_train


Ns_all = [100, 250, 500, 1000, 2000, 3000, 4000, 5000, 6000, 8000, 10000]
unet_types = ["mini", "medium", "maxi"]
tau_values=(1/6, 1/4, 1/3, 1/2, 2/3)
#tau_values = (0.10, 1 / 3, 0.50)
k_list = (2, 5, 10, 20, 50)
knn_k = 10
class_labels = [
"airplane",							
"automobile",									
"bird", 										
"cat", 										
"deer", 										
"dog", 										
"frog", 										
"horse", 										
"ship",								
"truck",
]
class_colors = ['#00429d', '#465db0', '#6d79c4', '#9097d7', '#b3b6eb', '#ffcab9', '#fd9291', '#e75d6f', '#c52a52', '#93003a']

class_colors = ['#00429d', '#9097d7', '#fd9291', '#c52a52', '#93003a']


In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def make_figure(Ns, unet_type, steps, save_path=None, classes=None):
    set_nice_params()

    fig = plt.figure(figsize=(5.5, 3.8), constrained_layout=False)

    gs = gridspec.GridSpec(
        nrows=len(Ns) + 2,
        ncols=3,
        figure=fig,
        width_ratios=[2., 1.0, 1.0],
        height_ratios=[0.23, 0.23] + [1.15] * len(Ns),
        wspace=0.23,
        hspace=0.68,
        left=0.04, 
        bottom=0.05, 
    )

    # --------------------------------------------------
    # Empty top-left space
    # --------------------------------------------------
    ax_blank = fig.add_subplot(gs[0:2, 0])
    ax_blank.remove()
    get_next_panel_label()

    # --------------------------------------------------
    # Top-right sparsity panel
    # --------------------------------------------------
    ax_sparsity = fig.add_subplot(gs[0:2, 1:])

    results_ref = get_data(max(Ns_all), "maxi", 1000000)

    plot_data_sparsity(
        ax_sparsity,
        results_ref,
        0,
        knn_k,
        bins=40,
    )

    ax_sparsity.set_title(
        get_next_panel_label() + r"Sparsity",
        fontsize=6,
        loc="left",
        pad=2,
    )
    ax_sparsity.set_xlabel("data sparsity", fontsize=6, labelpad=-8)

    ax_sparsity.tick_params(axis="both", labelsize=5, pad=1)
    #ax_sparsity.set_xticks([0,0.8])

    # --------------------------------------------------
    # Main rows
    # --------------------------------------------------
    for i, N in enumerate(Ns):
        row = i + 2

        # left image column starts below top block
        ax_img = fig.add_subplot(gs[row, 0])
        ax_dom = fig.add_subplot(gs[row, 1])
        ax_mem = fig.add_subplot(gs[row, 2])

        results = get_data(N, unet_type, steps)
        samples = load_samples(N, unet_type, steps)["samples"]

        plot_samples_with_closest_train(
            ax_img,
            results,
            samples,
            N,
            step=None,
            n=6,
            seed=0,
            ncols=6,
            sort_by="random",
            normalize=True,
            title=True,
            ylabel=True
        )

        plot_local_dominance(
            ax_dom,
            results,
            N,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
        )
        ax_dom.set_xscale("log")


        plot_memo_tau(
            ax_mem,
            results,
            N,
            tau_values=tau_values,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
            bins=10,
        )
        ax_mem.set_xscale("log")
        #ax_mem.set_xticks([1e-1, 1])

        ax_mem.set_title(
            get_next_panel_label() + "Memorization",
            loc="left",
            fontsize=6,
            pad=1,
        )

        for a in [ax_dom, ax_mem]:
            a.tick_params(axis="both", labelsize=5, pad=1)
            a.xaxis.labelpad = 1
            a.yaxis.labelpad = 1

        if i < len(Ns) - 1:
            ax_dom.set_xlabel("")
            ax_mem.set_xlabel("")

        if i == 0: 
            ax_mem.legend(frameon=False, fontsize=5,
                    ncol=2,
                    handlelength=0.9,
                    columnspacing=0.5,
                    borderaxespad=0.1,
                    loc="lower right",)
            ax_dom.legend(frameon=False, fontsize=6)


    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.025,
        )

    return fig
Ns_per_unet = [[100, 500, 1000], 
               [ 1000, 3000, 5000], 
               [ 4000, 5000, 8000], 
                [1000,10000, 20000]]

for Ns, unet_type, steps in zip(

    Ns_per_unet,
    ["mini", "medium", "maxi", "maxi"],
    [100000, 100000, 100000, 1000000],
):
    fig = make_figure(
        Ns,
        unet_type,
        steps,
        save_path=f"local_memo_{unet_type}_{steps}.pdf",
    )
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def make_classwise_figure(Ns, unet_type, steps, save_path=None, classes=None):
    set_nice_params()

    fig = plt.figure(figsize=(5.5, 3.8), constrained_layout=False)

    gs = gridspec.GridSpec(
        nrows=len(Ns) + 2,
        ncols=3,
        figure=fig,
        width_ratios=[2.5, 1.0, 1.0],
        height_ratios=[0.55, 0.55] + [1.15] * len(Ns),
        wspace=0.23,
        hspace=0.58,
        left=0.01, 
    )

    # --------------------------------------------------
    # Empty top-left space
    # --------------------------------------------------
    ax_blank = fig.add_subplot(gs[0:2, 0])
    ax_blank.axis("off")
    get_next_panel_label()

    # --------------------------------------------------
    # Top-right sparsity panel
    # --------------------------------------------------
    ax_sparsity = fig.add_subplot(gs[0:2, 1:])

    results_ref = get_data(max(Ns_all), "maxi", 1000000)

    plot_data_sparsity_per_class(
        ax_sparsity,
        results_ref,
        0,
        knn_k,
        colors=class_colors,
        class_labels=[class_labels[c] for c in classes],
        bins=20,
        classes=classes
    )

    ax_sparsity.set_title(
        get_next_panel_label() + r"Sparsity with $P$",
        fontsize=6,
        loc="left",
        pad=2,
    )
    ax_sparsity.set_xlabel("data sparsity", fontsize=6, labelpad=-8)

    ax_sparsity.tick_params(axis="both", labelsize=5, pad=1)
    #ax_sparsity.set_xticks([0,0.8])

    ax_sparsity.legend(
        frameon=False,
        fontsize=5,
        ncol=1,
        handlelength=0.9,
        columnspacing=0.5,
        borderaxespad=0.1,
        loc="upper right",
    )

    # --------------------------------------------------
    # Main rows
    # --------------------------------------------------
    for i, N in enumerate(Ns):
        row = i + 2

        # left image column starts below top block
        ax_img = fig.add_subplot(gs[row, 0])
        ax_dom = fig.add_subplot(gs[row, 1])
        ax_mem = fig.add_subplot(gs[row, 2])

        results = get_data(N, unet_type, steps)
        samples = load_samples(N, unet_type, steps)["samples"]

        plot_samples_with_closest_train(
            ax_img,
            results,
            samples,
            N,
            step=None,
            n=8,
            seed=0,
            ncols=10,
            sort_by="closest_label",
            balanced_by_closest_label=True,
            k_per_class=2,
            normalize=True,
            ylabel=True,
            title=True,
            num_classes=4,
            classes=classes
        )

        plot_local_dominance_per_class(
            ax_dom,
            results,
            N,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
            colors=class_colors,
            class_labels=[class_labels[c] for c in classes],
            bins=10,
            classes=classes
        )
        ax_dom.set_xscale("log")
        #ax_dom.set_xticks([1e-1, 1])
        ax_dom.set_title(
            get_next_panel_label() + rf"Loc. dominance",
            loc="left",
            fontsize=6,
            pad=1,
        )

        plot_memo_tau_per_class(
            ax_mem,
            results,
            N,
            1 / 3.0,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
            colors=class_colors,
            class_labels=[class_labels[c] for c in classes],
            bins=10,
            classes=classes
        )
        ax_mem.set_xscale("log")
        #ax_mem.set_xticks([1e-1, 1])

        ax_mem.set_title(
            get_next_panel_label() + "Memorization",
            loc="left",
            fontsize=6,
            pad=1,
        )

        for a in [ax_dom, ax_mem]:
            a.tick_params(axis="both", labelsize=5, pad=1)
            a.xaxis.labelpad = 1
            a.yaxis.labelpad = 1

        if i < len(Ns) - 1:
            ax_dom.set_xlabel("")
            ax_mem.set_xlabel("")

    fig.subplots_adjust(
        left=0.055,
        right=0.985,
        bottom=0.095,
        top=0.965,
    )

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.025,
        )

    return fig

Ns_per_unet = [[100, 500, 1000], 
               [ 1000, 3000, 5000], 
               [ 4000, 5000, 8000], 
                [5000,10000, 20000]]

for Ns, unet_type, steps in zip(
    Ns_per_unet,
    ["mini", "medium", "maxi", "maxi"],
    [100000, 100000, 100000, 1000000],
):
    fig = make_classwise_figure(
        Ns,
        unet_type,
        steps,
        save_path=f"cifar_classwise_memo_{unet_type}_{steps}.pdf",
        classes=[0, 8, 3, 9, 1],
    )
    plt.close(fig)

In [ ]:
def make_classwise_figure_full_top(
    Ns,
    unet_type,
    steps,
    save_path=None,
    classes=None,
    N0=1000,
):
    set_nice_params()

    fig = plt.figure(figsize=(5.5, 3.8), constrained_layout=False)

    gs = gridspec.GridSpec(
        nrows=len(Ns) + 2,
        ncols=3,
        figure=fig,
        width_ratios=[2.5, 1.0, 1.0],
        height_ratios=[0.55, 0.55] + [1.15] * len(Ns),
        wspace=0.23,
        hspace=0.58,
        left=0.055,
        right=0.985,
        bottom=0.095,
        top=0.965,
    )

    # --------------------------------------------------
    # Top-left panel: example generated/train matches
    # --------------------------------------------------
    ax_top_img = fig.add_subplot(gs[0:2, 0])

    results0 = get_data(N0, unet_type, steps)
    samples0 = load_samples(N0, unet_type, steps)["samples"]

    plot_samples_with_closest_train(
        ax_top_img,
        results0,
        samples0,
        N0,
        step=None,
        n=8,
        seed=0,
        ncols=8,
        sort_by="random",
        normalize=True,
        ylabel=True,
        title=True,
    )

    # --------------------------------------------------
    # Top-right sparsity panel
    # --------------------------------------------------
    ax_sparsity = fig.add_subplot(gs[0:2, 1:])

    results_ref = get_data(max(Ns_all), "maxi", 1000000)

    plot_data_sparsity_per_class(
        ax_sparsity,
        results_ref,
        0,
        knn_k,
        colors=class_colors,
        class_labels=[class_labels[c] for c in classes],
        bins=20,
        classes=classes,
    )

    ax_sparsity.set_title(
        get_next_panel_label() + r"Sparsity per class",
        fontsize=6,
        loc="left",
        pad=2,
    )
    ax_sparsity.set_xlabel("data sparsity", fontsize=6, labelpad=-8)
    ax_sparsity.tick_params(axis="both", labelsize=5, pad=1)
    #ax_sparsity.set_xticks([0, 0.8])

    ax_sparsity.legend(
        frameon=False,
        fontsize=5,
        ncol=1,
        handlelength=0.9,
        columnspacing=0.5,
        borderaxespad=0.1,
        loc="upper right",
    )

    # --------------------------------------------------
    # Main rows
    # --------------------------------------------------
    for i, N in enumerate(Ns):
        row = i + 2

        ax_img = fig.add_subplot(gs[row, 0])
        ax_dom = fig.add_subplot(gs[row, 1])
        ax_mem = fig.add_subplot(gs[row, 2])

        results = get_data(N, unet_type, steps)
        samples = load_samples(N, unet_type, steps)["samples"]

        plot_samples_with_closest_train(
            ax_img,
            results,
            samples,
            N,
            step=None,
            n=8,
            seed=i + 1,
            ncols=8,
            sort_by="random",
            normalize=True,
            ylabel=True,
            title=True,
        )

        plot_local_dominance_per_class(
            ax_dom,
            results,
            N,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
            colors=class_colors,
            class_labels=[class_labels[c] for c in classes],
            bins=10,
            classes=classes,
        )
        ax_dom.set_xscale("log")
       #ax_dom.set_xticks([1e-1, 1])
        ax_dom.set_title(
            get_next_panel_label() + "Loc. dominance",
            loc="left",
            fontsize=6,
            pad=1,
        )

        plot_memo_tau_per_class(
            ax_mem,
            results,
            N,
            1 /3.0,
            knn_k=knn_k,
            step=None,
            metric="mean_k",
            colors=class_colors,
            class_labels=[class_labels[c] for c in classes],
            bins=10,
            classes=classes,
        )
        ax_mem.set_xscale("log")
        #ax_mem.set_xticks([1e-1, 1])
        ax_mem.set_title(
            get_next_panel_label() + "Memorization",
            loc="left",
            fontsize=6,
            pad=1,
        )

        for a in [ax_dom, ax_mem]:
            a.tick_params(axis="both", labelsize=5, pad=1)
            a.xaxis.labelpad = 1
            a.yaxis.labelpad = 1

        if i < len(Ns) - 1:
            ax_dom.set_xlabel("")
            ax_mem.set_xlabel("")

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.025,
        )

    return fig

Ns_per_unet = [[100, 500, 1000], 
               [ 1000, 3000, 5000], 
               [ 4000, 5000, 8000], 
                [5000,10000, 20000]]

for Ns, unet_type, steps in zip(
    Ns_per_unet,
    ["mini", "medium", "maxi", "maxi"],
    [100000, 100000, 100000, 1000000],
):
    fig = make_classwise_figure_full_top(
        Ns,
        unet_type,
        steps,
        save_path=f"xcifar_classwise_memo_{unet_type}_{steps}.pdf",
        classes=[0, 8, 3, 1,],
    )
    plt.close(fig)